# Neo4j and Python

## Overview
Two main Python libraries for Neo4j:

| Library | Maintained by | Style | Use when |
|---|---|---|---|
| `neo4j` | Neo4j (official) | Low-level; explicit sessions and transactions | Production; async; full control |
| `py2neo` | Community | High-level OGM; Node/Relationship objects | Prototyping; simpler scripts |

**neo4j-driver connection string formats:**
```python
"bolt://localhost:7687"           # single instance
"neo4j://localhost:7687"          # routing (clusters)
"bolt+s://host:7687"              # TLS single instance
"neo4j+s://host:7687"             # TLS routing (production)
```

**Session pattern (neo4j-driver):**
```python
with driver.session() as session:
    result = session.execute_read(lambda tx: tx.run(cypher, **params).data())
```

**networkx note:** No Neo4j available offline. Code in this notebook is annotated and display-only for the driver/py2neo sections; the results section includes a live networkx demo.

---

In [1]:
print("=== neo4j-driver: official Python driver ===")
print()

driver_code = """
# pip install neo4j
from neo4j import GraphDatabase

# ── Connect ──────────────────────────────────────────────────────
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password"),
    max_connection_pool_size=50,
    connection_timeout=30,
)

# ── Run a read query ─────────────────────────────────────────────
def get_trial_team(driver, trial_id):
    with driver.session(database="neo4j") as session:
        result = session.run(
            '''
            MATCH (r:Researcher)-[:LEADS|CONTRIBUTES_TO]->(t:Trial {id: $trial_id})
            RETURN r.name AS name, r.role AS role
            ORDER BY r.role
            ''',
            trial_id=trial_id,
        )
        return [{"name": rec["name"], "role": rec["role"]} for rec in result]

team = get_trial_team(driver, "T001")
for member in team:
    print(f"  {member['role']:<20s}  {member['name']}")

# ── Run a write query ─────────────────────────────────────────────
def add_researcher(driver, researcher_id, name, role):
    with driver.session() as session:
        session.execute_write(
            lambda tx: tx.run(
                '''
                MERGE (r:Researcher {id: $id})
                ON CREATE SET r.name = $name, r.role = $role,
                              r.created_at = datetime()
                ON MATCH  SET r.updated_at = datetime()
                RETURN r
                ''',
                id=researcher_id, name=name, role=role,
            )
        )
add_researcher(driver, "R006", "Kofi Asante", "Researcher")

# ── Explicit transaction control ─────────────────────────────────
with driver.session() as session:
    with session.begin_transaction() as tx:
        tx.run("MERGE (r:Researcher {id: 'R007'}) SET r.name = 'Test'")
        # If anything fails after this point:
        tx.rollback()   # or tx.commit() on success
        # The session context manager auto-commits if no error

driver.close()
"""
print(driver_code)


=== neo4j-driver: official Python driver ===


# pip install neo4j
from neo4j import GraphDatabase

# ── Connect ──────────────────────────────────────────────────────
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password"),
    max_connection_pool_size=50,
    connection_timeout=30,
)

# ── Run a read query ─────────────────────────────────────────────
def get_trial_team(driver, trial_id):
    with driver.session(database="neo4j") as session:
        result = session.run(
            '''
            MATCH (r:Researcher)-[:LEADS|CONTRIBUTES_TO]->(t:Trial {id: $trial_id})
            RETURN r.name AS name, r.role AS role
            ORDER BY r.role
            ''',
            trial_id=trial_id,
        )
        return [{"name": rec["name"], "role": rec["role"]} for rec in result]

team = get_trial_team(driver, "T001")
for member in team:
    print(f"  {member['role']:<20s}  {member['name']}")

# ── Run a write query ──────────────────────────────────

---
## py2neo: higher-level interface

In [2]:
print("=== py2neo: higher-level Neo4j Python library ===")
print()

py2neo_code = """
# pip install py2neo
from py2neo import Graph, Node, Relationship
from py2neo.ogm import GraphObject, Property, RelatedTo

# ── Connect ──────────────────────────────────────────────────────
graph = Graph("bolt://localhost:7687", auth=("neo4j", "password"))

# ── py2neo Node and Relationship objects ────────────────────────
r = Node("Researcher", id="R001", name="Aroha Ngata", role="PI")
t = Node("Trial", id="T001", title="Cardio Alpha", phase=2)
leads = Relationship(r, "LEADS", t, role="Principal Investigator")
graph.create(leads)   # creates both nodes + relationship

# ── py2neo OGM (Object Graph Mapper) ─────────────────────────────
class Researcher(GraphObject):
    __primarykey__ = "id"
    id    = Property()
    name  = Property()
    role  = Property()
    leads = RelatedTo("Trial", "LEADS")
    contributes_to = RelatedTo("Trial", "CONTRIBUTES_TO")

# Push a Researcher to the database
r_obj = Researcher()
r_obj.id   = "R010"
r_obj.name = "New Researcher"
r_obj.role = "Researcher"
graph.push(r_obj)

# Query via OGM
aroha = Researcher.match(graph, "R001").first()
print(f"Name: {aroha.name}, role: {aroha.role}")
for trial in aroha.leads:
    print(f"  Leads: {trial.title}")

# ── Raw Cypher via py2neo ────────────────────────────────────────
results = graph.run(
    "MATCH (r:Researcher)-[:LEADS]->(t:Trial) "
    "WHERE t.phase >= $min_phase "
    "RETURN r.name, t.title, t.phase",
    min_phase=2
).data()
for row in results:
    print(row)
"""
print(py2neo_code)

print("=== Driver comparison ===")
comparison = [
    ("neo4j-driver", "Official Bolt driver",
     "Low-level, full control, explicit tx management, async support"),
    ("py2neo",       "Community high-level library",
     "OGM, Node/Relationship objects, simpler for prototyping"),
    ("neo4j-driver", "Production recommendation",
     "Better maintained, official support, async-compatible"),
    ("neomodel",     "Django-like OGM",
     "Similar to py2neo OGM; used in Flask/Django apps"),
]
print(f"\n  {'Library':<14s}  {'Type':<26s}  Notes")
print("  " + "-"*75)
for lib, typ, notes in comparison:
    print(f"  {lib:<14s}  {typ:<26s}  {notes}")


=== py2neo: higher-level Neo4j Python library ===


# pip install py2neo
from py2neo import Graph, Node, Relationship
from py2neo.ogm import GraphObject, Property, RelatedTo

# ── Connect ──────────────────────────────────────────────────────
graph = Graph("bolt://localhost:7687", auth=("neo4j", "password"))

# ── py2neo Node and Relationship objects ────────────────────────
r = Node("Researcher", id="R001", name="Aroha Ngata", role="PI")
t = Node("Trial", id="T001", title="Cardio Alpha", phase=2)
leads = Relationship(r, "LEADS", t, role="Principal Investigator")
graph.create(leads)   # creates both nodes + relationship

# ── py2neo OGM (Object Graph Mapper) ─────────────────────────────
class Researcher(GraphObject):
    __primarykey__ = "id"
    id    = Property()
    name  = Property()
    role  = Property()
    leads = RelatedTo("Trial", "LEADS")
    contributes_to = RelatedTo("Trial", "CONTRIBUTES_TO")

# Push a Researcher to the database
r_obj = Researcher()
r_obj.id   = "R010"
r

---
## Result handling, data types, and live demo

In [3]:
import networkx as nx

print("=== Result handling and data type patterns ===")
print()

result_code = """
# neo4j-driver result handling
from neo4j import GraphDatabase, Result

def get_researchers_with_trials(driver):
    with driver.session() as session:
        result = session.run('''
            MATCH (r:Researcher)-[:LEADS|CONTRIBUTES_TO]->(t:Trial)
            WITH  r, collect(t.title) AS trials
            RETURN r.name AS name, r.role AS role, trials, size(trials) AS n_trials
            ORDER BY n_trials DESC
        ''')
        # result.data() -> list of dicts
        return result.data()

rows = get_researchers_with_trials(driver)
for row in rows:
    print(f"  {row['name']:<20s}  n_trials={row['n_trials']}  {row['trials']}")

# Accessing node/relationship objects (not just scalars)
def get_full_nodes(driver):
    with driver.session() as session:
        result = session.run("MATCH (r:Researcher) RETURN r LIMIT 3")
        for record in result:
            node = record["r"]         # neo4j.graph.Node object
            print(f"  id={node['id']}  labels={list(node.labels)}  props={dict(node)}")

# Data types in Neo4j properties:
types_table = [
    ("Integer",   "Python int",     "r.h_index = 12"),
    ("Float",     "Python float",   "r.citation_rate = 4.7"),
    ("String",    "Python str",     "r.name = 'Aroha Ngata'"),
    ("Boolean",   "Python bool",    "t.active = True"),
    ("Date",      "neo4j.time.Date","r.dob = date('1985-03-15')"),
    ("DateTime",  "neo4j.time.DateTime","r.created_at = datetime()"),
    ("List",      "Python list",    "r.languages = ['Python', 'R', 'SQL']"),
    ("Null",      "Python None",    "r.optional_field = null"),
    ("Point",     "neo4j.spatial.Point","r.location = point({longitude: -63.5, latitude: 44.6})"),
]
print("\\nNeo4j property data types:")
print(f"  {'Neo4j type':<12s}  {'Python type':<24s}  Example")
print("  " + "-"*65)
for neo_type, py_type, example in types_table:
    print(f"  {neo_type:<12s}  {py_type:<24s}  {example}")
"""
print(result_code)

# Live demo: simulate result processing with networkx
G_demo = nx.MultiDiGraph()
G_demo.add_node("R001", label="Researcher", name="Aroha Ngata", role="PI")
G_demo.add_node("R002", label="Researcher", name="Liam Chen", role="Biostatistician")
G_demo.add_node("T001", label="Trial", title="Cardio Alpha")
G_demo.add_node("T002", label="Trial", title="Neuro Beta")
G_demo.add_edge("R001","T001",rel_type="LEADS")
G_demo.add_edge("R002","T001",rel_type="CONTRIBUTES_TO")
G_demo.add_edge("R002","T002",rel_type="CONTRIBUTES_TO")

print("\nSimulated result: researchers with trial lists")
print(f"  {'name':<22s}  {'role':<20s}  trials")
print("  " + "-"*65)
for rid, rdata in G_demo.nodes(data=True):
    if rdata.get('label') == 'Researcher':
        trials = [G_demo.nodes[v]['title']
                  for u, v, ed in G_demo.edges(data=True)
                  if u == rid and ed['rel_type'] in ('LEADS','CONTRIBUTES_TO')]
        print(f"  {rdata['name']:<22s}  {rdata['role']:<20s}  {trials}")


=== Result handling and data type patterns ===


# neo4j-driver result handling
from neo4j import GraphDatabase, Result

def get_researchers_with_trials(driver):
    with driver.session() as session:
        result = session.run('''
            MATCH (r:Researcher)-[:LEADS|CONTRIBUTES_TO]->(t:Trial)
            WITH  r, collect(t.title) AS trials
            RETURN r.name AS name, r.role AS role, trials, size(trials) AS n_trials
            ORDER BY n_trials DESC
        ''')
        # result.data() -> list of dicts
        return result.data()

rows = get_researchers_with_trials(driver)
for row in rows:
    print(f"  {row['name']:<20s}  n_trials={row['n_trials']}  {row['trials']}")

# Accessing node/relationship objects (not just scalars)
def get_full_nodes(driver):
    with driver.session() as session:
        result = session.run("MATCH (r:Researcher) RETURN r LIMIT 3")
        for record in result:
            node = record["r"]         # neo4j.graph.Node object
            print(f

---
## Common Pitfalls

**1. Not closing the driver**
`GraphDatabase.driver(...)` creates a connection pool. Not calling `driver.close()` at application shutdown leaks connections. Use `with GraphDatabase.driver(...) as driver:` or call `driver.close()` in application teardown.

**2. Using f-strings instead of parameters**
`session.run(f"MATCH (r:Researcher {{name: '{name}'}})")` is a Cypher injection vulnerability and bypasses the query plan cache. Always pass parameters: `session.run('MATCH (r:Researcher {name: $name})', name=name)`.

**3. Holding sessions open for long operations**
A `Session` holds a network connection. Opening a session, doing Python computation for 30 seconds, then running a query holds a connection from the pool idle. Open sessions only when needed; close them immediately after the query completes.

**4. Not retrying on transient errors**
Neo4j can raise `TransientError` (e.g. deadlock detected) for write transactions. The neo4j-driver's `execute_write()` and `execute_read()` automatically retry on transient errors. Using raw `session.run()` outside these helpers requires manual retry.

**5. py2neo OGM loading entire relationship collections unnecessarily**
Accessing `researcher.leads` on a py2neo OGM object loads all related Trial nodes immediately. For nodes with hundreds of relationships, this is a full traversal. Use explicit Cypher with `LIMIT` instead of traversing OGM relationship collections.

**6. Assuming bolt:// and neo4j:// are interchangeable**
`bolt://` connects to a single instance. `neo4j://` uses the routing protocol and connects to a Neo4j cluster, load-balancing reads across replicas. In production clusters, always use `neo4j://` (or `neo4j+s://` for TLS).


---
*sql_methods_library - Samantha McGarrigle*